# Fresh dev v17 generator: coursework data evidence

Read-only analysis of the published historical source, not a new generation run or Bronze/Silver acceptance.
All inventory-listed files are included; there is no row sample. Outputs are cleared and fresh dev execution is pending.

From `src/data-pipeline/simulator`, start `uv run --locked --group notebook jupyter lab`.
The source selects fresh dev v17, using a dev-scoped local copy if available. Set its actual manifest digest after publication. Otherwise supply the existing S3 read credentials
through the environment before starting Jupyter. Credentials are never displayed or stored in the notebook.

The configuration, generator fingerprint, manifest hash and query code below identify this evidence.
DuckDB is used for source inspection; Spark transformations and optimization evidence belong to DP1/DP2.

In [ ]:
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlsplit

import boto3
import duckdb
import matplotlib.pyplot as plt
import pyarrow as pa
import yaml

%matplotlib inline

SOURCE_URI = "s3://platform-dev/datasets/payment-gateway/historical/v17"
EXPECTED_MANIFEST_SHA256 = None
LOCAL_COPY = Path.home() / ".cache/fraud-coursework/dev/v17"
DATASET = str(LOCAL_COPY) if (LOCAL_COPY / "manifest.json").exists() else SOURCE_URI
ENDPOINT = "https://minio.home.phuchoang.sbs"

if EXPECTED_MANIFEST_SHA256 is None:
    raise ValueError("Set EXPECTED_MANIFEST_SHA256 after fresh dev v17 publication")

db = duckdb.connect()
db.execute("SET TimeZone = 'UTC'")
db.execute("SET threads = 2")
db.execute("SET memory_limit = '2GB'")
if DATASET.startswith("s3://"):
    parsed = urlsplit(DATASET)
    client = boto3.client("s3", endpoint_url=ENDPOINT)
    manifest_bytes = client.get_object(Bucket=parsed.netloc, Key=parsed.path.lstrip("/") + "/manifest.json")["Body"].read()
    db.execute("CREATE SECRET (TYPE s3, PROVIDER credential_chain, ENDPOINT 'minio.home.phuchoang.sbs', URL_STYLE 'path', REGION 'us-east-1')")
else:
    manifest_bytes = (Path(DATASET) / "manifest.json").read_bytes()
manifest_sha256 = hashlib.sha256(manifest_bytes).hexdigest()
if manifest_sha256 != EXPECTED_MANIFEST_SHA256:
    raise ValueError("The manifest differs from the selected published v17 identity")
manifest = json.loads(manifest_bytes)
print(json.dumps({
    "captured_at": datetime.now(timezone.utc).isoformat(),
    "source_uri": SOURCE_URI,
    "read_location": DATASET,
    "manifest_sha256": manifest_sha256,
    "config_sha256": manifest["config_sha256"],
    "generator_fingerprint": manifest["generator_revision"],
    "duckdb_version": duckdb.__version__,
    "source_validation": manifest["validation"],
}, indent=2))

## Configuration and physical volume

This is the configuration retained in the published manifest, not today's editable YAML.
Counts are configured entity pools, not measured cardinalities. The duplicate-rate setting is extra identical deliveries
as a fraction of all event deliveries. Byte counts are complete physical Parquet object sizes.

In [ ]:
print(yaml.safe_dump(manifest["configuration"], sort_keys=False))
print("Schema versions:", manifest["schema_versions"])
for table in ("payment_events", "fraud_labels"):
    print(table, json.dumps(manifest["storage"][table], indent=2))
print("Format:", manifest["storage"]["format"], "compression:", manifest["storage"]["compression"], "level:", manifest["storage"]["compression_level"])

## Read exactly the published inventory

Read mixed schemas by name so old optional fields become null. Physical schema versions come from the inventory;
`schema_version` inside an event remains the producer's envelope version. Local copies are checked against every
inventory file's SHA256 and physical size before querying. Remote analysis pins the manifest and reads its exact files;
DP1 owns full remote integrity acceptance.

In [ ]:
files = {table: [item for item in manifest["inventory"] if item["table"] == table] for table in ("payment_events", "fraud_labels")}
if not DATASET.startswith("s3://"):
    for item in manifest["inventory"]:
        body = (Path(DATASET) / item["key"]).read_bytes()
        if len(body) != item["bytes"] or hashlib.sha256(body).hexdigest() != item["sha256"]:
            raise ValueError("Inventory file differs: " + item["key"])
    print("All", len(manifest["inventory"]), "local inventory files match their published sizes and hashes")
for table, entries in files.items():
    paths = [DATASET + "/" + item["key"] for item in entries]
    db.from_parquet(paths, union_by_name=True, hive_partitioning=False, filename=True).create_view(table)
file_versions = [{"path": DATASET + "/" + item["key"], "physical_schema_version": item["schema_version"]} for item in files["payment_events"]]
db.register("file_versions", pa.Table.from_pylist(file_versions))
db.sql("""
    CREATE TEMP TABLE events AS
    SELECT e.* EXCLUDE (filename), v.physical_schema_version
    FROM (SELECT *, filename FROM payment_events) e JOIN file_versions v ON e.filename = v.path
""")
db.sql("CREATE TEMP TABLE event_variants AS SELECT DISTINCT * FROM events")
db.sql("""
    CREATE TEMP TABLE conflicting_event_ids AS
    SELECT integration_id, event_id FROM event_variants
    GROUP BY integration_id, event_id HAVING count(*) > 1
""")
db.sql("""
    CREATE TEMP VIEW canonical_events AS
    SELECT e.* FROM event_variants e ANTI JOIN conflicting_event_ids USING (integration_id, event_id)
""")
db.sql("""
    SELECT 'payment_events' AS port, count(*) AS observed_rows FROM events
    UNION ALL SELECT 'fraud_labels', count(*) FROM fraud_labels
""").show()

## Duplicate deliveries before and after illustrative deduplication

An exact variant includes all normalized business fields and physical schema identity; file locations are excluded.
Only identical variants collapse. Conflicting scoped event identities are counted and excluded, rather than selecting
an arbitrary winner. The before-dedup rate denominator is all raw event deliveries; the after-dedup chart uses canonical output rows. This source analysis does not establish
persisted Silver acceptance; the actual Spark pipeline retains delivery provenance.

In [ ]:
duplicates = db.sql("""
    WITH totals AS (
      SELECT (SELECT count(*) FROM events) AS raw_deliveries,
             (SELECT count(*) FROM event_variants) AS distinct_variants,
             (SELECT count(*) FROM conflicting_event_ids) AS conflicting_identities,
             (SELECT count(*) FROM canonical_events) AS canonical_events
    )
    SELECT *, raw_deliveries - distinct_variants AS identical_extra_deliveries,
           round(100.0 * (raw_deliveries - distinct_variants) / raw_deliveries, 6) AS duplicate_pct_before,
           (SELECT coalesce(sum(n - 1), 0) FROM (
               SELECT count(*) AS n FROM canonical_events GROUP BY integration_id, event_id HAVING count(*) > 1
           )) AS duplicate_deliveries_after
    FROM totals
""")
duplicates.show(max_width=180)
raw, variants, conflicts, canonical, extra, rate, after = duplicates.fetchone()
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(["Before", "After illustrative dedup"], [float(rate), 100 * after / canonical if canonical else 0])
ax.set(ylabel="Extra deliveries (%)", title="Identical event delivery duplicates")
plt.tight_layout()
plt.show()

## Skew: merchant, category and shipping region

Merchant and region percentages use canonical `payment_created` events. Category percentages use all canonical
checkout line items, not only the first item. The category denominator can therefore differ from the payment denominator.
Tables show the largest shares; charts do not imply uniform distributions were expected.

In [ ]:
db.sql("CREATE TEMP VIEW created AS SELECT integration_id, event_id, payment_created AS payment FROM canonical_events WHERE payment_created IS NOT NULL")
merchant = db.sql("""
    SELECT payment.merchant.merchant_id AS merchant, count(*) AS payments,
           round(100.0 * count(*) / sum(count(*)) OVER (), 3) AS pct_all_payments
    FROM created GROUP BY merchant ORDER BY payments DESC, merchant
""")
categories = db.sql("""
    SELECT item.category AS category, count(*) AS line_items,
           round(100.0 * count(*) / sum(count(*)) OVER (), 3) AS pct_all_line_items
    FROM created, UNNEST(payment.snapshot.items) AS items(item)
    GROUP BY category ORDER BY line_items DESC
""")
regions = db.sql("""
    SELECT payment.snapshot.shipping.region AS region, count(*) AS payments,
           round(100.0 * count(*) / sum(count(*)) OVER (), 3) AS pct_all_payments
    FROM created GROUP BY region ORDER BY payments DESC
""")
merchant.limit(10).show()
categories.show()
regions.show()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
rows = merchant.limit(10).fetchall()
axes[0].bar(range(1, len(rows) + 1), [float(row[2]) for row in rows])
axes[0].set(title="Top merchant shares", xlabel="Merchant rank", ylabel="All canonical payments (%)")
rows = categories.fetchall()
axes[1].barh([row[0] for row in rows][::-1], [float(row[2]) for row in rows][::-1])
axes[1].set(title="Category shares", xlabel="All canonical line items (%)")
plt.tight_layout()
plt.show()

## High cardinality: observed identifiers versus configured pools

Identifiers are scoped by integration. Approximate counts use DuckDB's `approx_count_distinct` (HyperLogLog);
it has no user-selected relative-error parameter here. Exact counts over this small development source provide
an observed error comparison, not a guaranteed error bound. These are not Spark `approx_count_distinct` results.
All counts exclude null identifiers and use canonical attempt-start events. Configured pools describe the available
entity world; observed distinct fingerprints need not exhaust or match those pools.

In [ ]:
db.sql("CREATE TEMP VIEW started AS SELECT integration_id, attempt_started AS attempt FROM canonical_events WHERE attempt_started IS NOT NULL")
identifiers = {
    "customers": "attempt.context.customer.customer_id",
    "merchants": "attempt.context.merchant.merchant_id",
    "devices": "attempt.signals.device_fingerprint",
    "ip_networks": "attempt.signals.ip_network_fingerprint",
    "payment_methods": "attempt.signals.payment_method.fingerprint",
    "addresses": "attempt.context.snapshot.shipping.address_fingerprint",
}
for name, field in identifiers.items():
    query = f"""
      SELECT '{name}' AS identifier, {manifest['configured']['cardinality'][name]} AS configured_pool,
        count(*) AS attempt_denominator, count({field}) AS nonnull_attempts,
        approx_count_distinct(CASE WHEN {field} IS NOT NULL THEN struct_pack(scope := integration_id, id := {field}) END) AS approximate_distinct,
        count(DISTINCT CASE WHEN {field} IS NOT NULL THEN struct_pack(scope := integration_id, id := {field}) END) AS exact_distinct
      FROM started
    """
    db.sql(f"SELECT *, round(100.0 * (approximate_distinct - exact_distinct) / nullif(exact_distinct, 0), 2) AS observed_error_pct FROM ({query})").show(max_width=170)

## Schema evolution: old absence versus new reference values

Only `risk_decision_recorded` rows are eligible for assessment references. Results use raw decision deliveries,
including identical duplicates, grouped by inventory physical version and UTC event date. Old physical absence is
reported separately from null/unavailable values in the new shape. The event-envelope version is shown independently.
Synthetic references do not prove Fraud-owned evidence export or online feature-vector availability.

In [ ]:
for version in manifest["schema_versions"]["payment_events"]:
    sample = next(item for item in files["payment_events"] if item["schema_version"] == version)
    physical = db.from_parquet(DATASET + "/" + sample["key"], hive_partitioning=False)
    print("Physical schema", version, physical.types[physical.columns.index("risk_decision_recorded")])

db.sql("""
    SELECT physical_schema_version, schema_version AS envelope_version, count(*) AS deliveries
    FROM events GROUP BY ALL ORDER BY ALL
""").show()
references = """
    SELECT physical_schema_version, CAST(occurred_at AS DATE) AS event_date,
           count(*) AS eligible_decision_deliveries,
           count(*) FILTER (WHERE risk_decision_recorded.assessment.assessment_id IS NULL) AS assessment_id_nulls,
           count(risk_decision_recorded.assessment.assessment_id) AS assessment_id_values,
           count(*) FILTER (WHERE risk_decision_recorded.assessment.evidence_sha256 IS NULL) AS evidence_hash_nulls,
           count(risk_decision_recorded.assessment.evidence_sha256) AS evidence_hash_values
    FROM events WHERE risk_decision_recorded IS NOT NULL GROUP BY ALL
"""
db.sql(f"""
    SELECT physical_schema_version, sum(eligible_decision_deliveries) AS eligible_decision_deliveries,
           sum(assessment_id_nulls) AS assessment_id_nulls, sum(assessment_id_values) AS assessment_id_values,
           sum(evidence_hash_nulls) AS evidence_hash_nulls, sum(evidence_hash_values) AS evidence_hash_values
    FROM ({references}) GROUP BY ALL ORDER BY ALL
""").show(max_width=170)
db.sql(f"SELECT * FROM ({references}) WHERE event_date BETWEEN DATE '2025-08-13' AND DATE '2025-08-17' ORDER BY event_date, physical_schema_version").show(max_width=170)
rows = db.sql(f"SELECT event_date, 100.0 * sum(assessment_id_nulls) / sum(eligible_decision_deliveries) FROM ({references}) GROUP BY event_date ORDER BY event_date").fetchall()
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot([row[0] for row in rows], [float(row[1]) for row in rows])
ax.set(title="Missing assessment references across the physical schema change", ylabel="Eligible decision deliveries (%)", xlabel="UTC event date")
plt.tight_layout()
plt.show()

## Event and label tails

Labels are independent observations. Chargebacks are positive reports; absence of a report means unknown.
Simulation truth can provide training targets, subject to availability time. Scenario/future-outcome columns must stay
out of predecision features. These source diagnostics do not claim drift or streaming burst/late/duplicate acceptance.

In [ ]:
db.sql("SELECT min(occurred_at) AS first_event, max(occurred_at) AS last_event FROM events").show()
db.sql("""
    SELECT source, count(*) AS observations, min(occurred_at) AS first_occurrence,
           max(occurred_at) AS last_occurrence, max(label_available_at) AS last_available,
           count(*) FILTER (WHERE label_available_at > CAST(? AS TIMESTAMPTZ)) AS available_after_event_tail
    FROM fraud_labels GROUP BY source
""", params=[manifest["history_end"]]).show(max_width=190)

## What this evidence establishes

The outputs above describe the published generator configuration, physical volume, skew, identifier cardinality,
mixed physical schemas and identical duplicate deliveries. Before/after deduplication is illustrative source SQL.
DP1 must separately accept persisted Bronze; DP2 must separately accept Spark-curated Silver and its conservation,
quality and governance results. Spark optimization measurements remain a separate pipeline task.